# Normalizing Flow
## RealNVP
RealNVP (Real-valued Non-Volume Preserving) is a type of normalizing flow model that allows for efficient density estimation and sampling. It is based on the idea of transforming a simple base distribution (like a Gaussian) into a more complex target distribution through a series of invertible transformations. The key feature of RealNVP is that it uses coupling layers, which split the input into two parts and apply transformations to one part conditioned on the other. This allows for efficient computation of the Jacobian determinant, which is necessary for calculating the likelihood of the data under the model.

In [ ]:
import torch
from torch import nn

The main components of a RealNVP model are:
1. **Coupling Layers**: These layers split the input into two parts. One part is transformed using a function that depends on the other part. This transformation is designed to be invertible, allowing for both forward and inverse passes through the model.
2. **Activation Normalization**: This component normalizes the activations of the network, which helps to stabilize the training process and improve the convergence of the model.
3. **Permutation Layers**: These layers permute the dimensions of the input data to ensure that all dimensions are transformed across multiple coupling layers, enhancing the expressiveness of the model.

#### Coupling Layers

In [ ]:
class Coupling(nn.Module):
    """A RealNVP-style coupling layer.

    Splits the input vector into two halves, uses the first to compute
    an affine transformation of the second, and then concatenates them
    back together.  This construction is invertible and the Jacobian
    is triangular so the log-determinant is easy to compute if needed.
    We do not use the log-det in the current training losses, but the
    architecture is a simple example of a discontinuous normalizing
    flow (the mapping is not defined by a differential equation).
    """

    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.dim = dim
        self.half = dim // 2
        self.second_half = dim - self.half
        # scale and translate nets operate on the first half
        self.scale_net = nn.Sequential(
            nn.Linear(self.half, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, self.second_half),
            nn.Tanh(),  # to keep the scaling factors bounded and prevent numerical instability
        )
        self.translate_net = nn.Sequential(
            nn.Linear(self.half, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, self.second_half),  # , bias=False
        )

        self.scale_s = nn.Parameter(torch.zeros(1, self.second_half))

        nn.init.zeros_(self.translate_net[-1].weight)
        nn.init.zeros_(self.translate_net[-1].bias)

    def forward(self, x):

        x1 = x[:, : self.half]
        x2 = x[:, self.half :]

        s = self.scale_net(x1) * self.scale_s
        t = self.translate_net(x1)

        # Affine Transformation: z2 = x2 * exp(s) + t
        y2 = x2 * torch.exp(s) + t

        # log determinant of the Jacobian
        log_det_J = s.sum(dim=-1)

        return torch.cat([x1, y2], dim=1), log_det_J

    def inverse(self, y):
        """Inverse of the coupling layer.

        Given output `y = [y1, y2]` where y2 = x2 * exp(s(y1)) + t(y1),
        recover x = [y1, (y2 - t) * exp(-s)]. Returns reconstructed x and
        the log-det of the inverse mapping (negative of forward log-det).
        """
        y1 = y[:, : self.half]
        y2 = y[:, self.half :]

        s = self.scale_net(y1) * self.scale_s
        t = self.translate_net(y1)

        x2 = (y2 - t) * torch.exp(-s)

        # log determinant of the Jacobian
        log_det_J = -s.sum(dim=-1)

        return torch.cat([y1, x2], dim=1), log_det_J


#### Activation Normalization

In [ ]:
class ActNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        # Learnable affine parameters
        self.loc = nn.Parameter(torch.zeros(1, dim))
        self.log_scale = nn.Parameter(torch.zeros(1, dim))

        # A flag to tell us if we have done the first-batch initialization
        self.register_buffer("initialized", torch.tensor(False))

    def forward(self, x):
        # Data-Dependent Initialization: Happens ONLY on the very first batch
        if not self.initialized and self.training:
            with torch.no_grad():
                # Calculate mean and standard deviation of the batch
                batch_mean = x.mean(dim=0, keepdim=True)
                batch_std = x.std(dim=0, unbiased=False, keepdim=True) + 1e-5

                # Set the parameters so that the output has mean=0, std=1
                self.loc.copy_(-batch_mean)
                self.log_scale.copy_(-torch.log(batch_std))

                # Mark as initialized so this NEVER runs again
                self.initialized.fill_(True)

        # Apply the Affine Transformation: Scale + Shift
        y = x * torch.exp(self.log_scale) + self.loc

        # Calculate Log Determinant
        log_det = torch.sum(self.log_scale).expand(x.shape[0])

        return y, log_det

    def inverse(self, y):
        # Reverse the Affine Transformation: Subtract Shift, then Divide by Scale
        x = (y - self.loc) * torch.exp(-self.log_scale)

        # Inverse log det is just the negative of the forward log det
        inv_log_det = torch.sum(-self.log_scale).expand(y.shape[0])

        return x, inv_log_det

#### Permutation Layers

In [ ]:
class Permute(nn.Module):
    """Simple layer that reverses the order of the input dimensions."""

    def forward(self, x):
        return x.flip(dims=[1]), torch.zeros(x.shape[0], device=x.device)

    def inverse(self, y):
        """Inverse of the permutation (same as forward for a flip)."""
        return y.flip(dims=[1]), torch.zeros(y.shape[0], device=y.device)

### RealNVPMap 
Once we have all the components, we can combine them into a RealNVP model. The RealNVPMap class will encapsulate the coupling layers, activation normalization, and permutation layers, allowing us to define a complete normalizing flow model.

In [ ]:
class NFMap(nn.Module):
    """Normalizing flow made of coupling layers and permutations."""

    def __init__(self, d, n_layers: int, hidden_dim: int):
        super().__init__()
        blocks = []
        self.dim = d
        for _ in range(n_layers):
            blocks.append(ActNorm(self.dim))
            blocks.append(Coupling(self.dim, hidden_dim))
            blocks.append(Permute())
        self.flow = nn.Sequential(*blocks)

    def forward(self, x):
        log_det_total = torch.zeros(x.shape[0], device=x.device).double()
        for _, module in enumerate(self.flow):
            x, log_det = module(x)
            log_det_total += log_det

        return x, log_det_total

    def inverse(self, z):
        """Apply the inverse flow: run modules in reverse and apply each
        module's inverse.
        """
        log_det_total = torch.zeros(z.shape[0], device=z.device).double()
        # iterate modules in reverse order
        for module in reversed(list(self.flow)):
            z, log_det = module.inverse(z)
            log_det_total = log_det_total + log_det

        return z, log_det_total

To be completed...